In [3]:
import torch
import torch.nn as nn

class Net(nn.Module):
      def __init__(self):
            super(Net, self).__init__()
            self.fc1=nn.Linear(10,5) #holds bias and weights
            self.bn=nn.BatchNorm1d(5) # holds bias , weights , running_mean , running_var , num_batches_tracked
            self.fc2=nn.Linear(5,3) #holds weights and bias
            self.drop=nn.Dropout(0.3) # make 30% numbers to 0.0
            self.fc3=nn.Linear(3,1)
            self.register_buffer("scale",torch.tensor(1.0))

model=Net()

- Parameters

In [4]:
#parametrs in the model
for p in model.named_parameters():
  print(p)

('fc1.weight', Parameter containing:
tensor([[ 0.1705,  0.1730, -0.0968,  0.1330,  0.0481,  0.1487,  0.1068,  0.0635,
          0.3029, -0.1317],
        [ 0.1391, -0.2146,  0.2578,  0.0843, -0.0756,  0.0159,  0.0667, -0.2904,
          0.0963,  0.0837],
        [-0.2289,  0.1233, -0.2243,  0.3055,  0.0118, -0.0883,  0.0951, -0.0209,
         -0.0549, -0.2797],
        [-0.0059, -0.1420,  0.0608, -0.1571,  0.2542, -0.1747, -0.2639, -0.3068,
          0.0194,  0.1006],
        [ 0.1919,  0.0499,  0.2276, -0.0375, -0.2149,  0.1826,  0.0040, -0.0760,
          0.0339, -0.2139]], requires_grad=True))
('fc1.bias', Parameter containing:
tensor([ 0.2277,  0.2237, -0.0481,  0.2653,  0.0741], requires_grad=True))
('bn.weight', Parameter containing:
tensor([1., 1., 1., 1., 1.], requires_grad=True))
('bn.bias', Parameter containing:
tensor([0., 0., 0., 0., 0.], requires_grad=True))
('fc2.weight', Parameter containing:
tensor([[ 0.0395, -0.1864, -0.0178,  0.0599,  0.4291],
        [ 0.3627,  0.088

In [5]:
# Job 1: hand the learnable numbers to the optimizer
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
#Every single time your loop execution hits the optimizer's update line, a weight update occurs immediately right then and there.

# Job 2: count them
total = sum(p.numel() for p in model.parameters())
print(total)

87


In [6]:
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)  #parameters that are actively training
print(trainable)


87


In [7]:
# usecase of named_paramters() when we wnat any specific named parameter
head = [p for n, p in model.named_parameters() if n.startswith("fc2")]  # n->name and p->parameter
body = [p for n, p in model.named_parameters() if not n.startswith("fc2")]

opt = torch.optim.SGD([
    {"params": body, "lr": 1e-3},
    {"params": head, "lr": 1e-2},
])

- Buffers

In [8]:
# buffers
for b in model.named_buffers():
  print(b)

# Notice that BatchNorm's running averages are buffers. They get saved, they move to the GPU,
# but the optimizer never touches them. They're updated by BatchNorm itself during training.

('scale', tensor(1.))
('bn.running_mean', tensor([0., 0., 0., 0., 0.]))
('bn.running_var', tensor([1., 1., 1., 1., 1.]))
('bn.num_batches_tracked', tensor(0))


- Methods

In [9]:
print([c for c in model.named_children()]) #Only the direct pieces, one level down.

[('fc1', Linear(in_features=10, out_features=5, bias=True)), ('bn', BatchNorm1d(5, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)), ('fc2', Linear(in_features=5, out_features=3, bias=True)), ('drop', Dropout(p=0.3, inplace=False)), ('fc3', Linear(in_features=3, out_features=1, bias=True))]


In [10]:
print([ m for m in model.named_modules()]) # all the modules at any depth
# in o/p ->" "" defines the model module itself becoz it's itself a module of nn.Module

[('', Net(
  (fc1): Linear(in_features=10, out_features=5, bias=True)
  (bn): BatchNorm1d(5, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc2): Linear(in_features=5, out_features=3, bias=True)
  (drop): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=3, out_features=1, bias=True)
)), ('fc1', Linear(in_features=10, out_features=5, bias=True)), ('bn', BatchNorm1d(5, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)), ('fc2', Linear(in_features=5, out_features=3, bias=True)), ('drop', Dropout(p=0.3, inplace=False)), ('fc3', Linear(in_features=3, out_features=1, bias=True))]


In [11]:
buffers = [b for n, b in model.named_buffers() if isinstance(b, torch.Tensor)]
print(buffers)

[tensor(1.), tensor([0., 0., 0., 0., 0.]), tensor([1., 1., 1., 1., 1.]), tensor(0)]


In [12]:
# module vs children
net = nn.Sequential(
    nn.Sequential(nn.Linear(2, 2), nn.ReLU()),   # a block
    nn.Linear(2, 1),
)

print((list(net.children())))   # 2   → the block, and the last Linear
print("\n")
print((list(net.modules())))    # 5   → net, block, Linear, ReLU, Linear

[Sequential(
  (0): Linear(in_features=2, out_features=2, bias=True)
  (1): ReLU()
), Linear(in_features=2, out_features=1, bias=True)]


[Sequential(
  (0): Sequential(
    (0): Linear(in_features=2, out_features=2, bias=True)
    (1): ReLU()
  )
  (1): Linear(in_features=2, out_features=1, bias=True)
), Sequential(
  (0): Linear(in_features=2, out_features=2, bias=True)
  (1): ReLU()
), Linear(in_features=2, out_features=2, bias=True), ReLU(), Linear(in_features=2, out_features=1, bias=True)]


In [13]:
# modules
linears=[m for m in model.modules() if isinstance(m,nn.Linear)] #check that how much modules are Linear
print(linears)

[Linear(in_features=10, out_features=5, bias=True), Linear(in_features=5, out_features=3, bias=True), Linear(in_features=3, out_features=1, bias=True)]


# Instance

In [14]:
#isinstance() is used to check if a specific layer or module belongs to a particular class type.
for m in model.modules():
    if isinstance(m, nn.Linear):
        print(f"{m} -> YES! This is a linear layer. Let it pass.")
    else:
        print(f"{m} -> NO. Ignore this layer.")

Net(
  (fc1): Linear(in_features=10, out_features=5, bias=True)
  (bn): BatchNorm1d(5, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc2): Linear(in_features=5, out_features=3, bias=True)
  (drop): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=3, out_features=1, bias=True)
) -> NO. Ignore this layer.
Linear(in_features=10, out_features=5, bias=True) -> YES! This is a linear layer. Let it pass.
BatchNorm1d(5, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True) -> NO. Ignore this layer.
Linear(in_features=5, out_features=3, bias=True) -> YES! This is a linear layer. Let it pass.
Dropout(p=0.3, inplace=False) -> NO. Ignore this layer.
Linear(in_features=3, out_features=1, bias=True) -> YES! This is a linear layer. Let it pass.


## Usecases of instance()
1. Initializing Weights Dynamically


In [16]:
import torch.nn as nn

for module in model.modules():
    # If the layer is a Linear layer, initialize weights with Xavier uniform
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight) #nn.init.xavier_uniform_(module.weight) fills the weights of your Linear layer with values
        # calculated using the Xavier (Glorot) Uniform initialization method.
        if module.bias is not None:
            nn.init.zeros_(module.bias)



2. Freezing Specific Layer Types


In [ ]:
for module in model.modules():
    # Find all Batch Normalization layers
    if isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d)):
        module.eval()  # Set them to evaluation mode
        for param in module.parameters():
            param.requires_grad = False  # Freeze their parameters


In [15]:
model.get_submodule("fc1")            # the Linear layer
model.get_parameter("fc1.weight")     # the weight tensor
model.get_buffer("bn.running_mean")   # the buffer

tensor([0., 0., 0., 0., 0.])